## 1. Импорт библиотек и пути к данным

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RAW_DATA_PATH = Path("D:\\Iowa_Liquor_Sales.csv")
DATA_DIR = Path("data/processed")

# Границы сохранены по логике исходного ноутбука: обе границы строгие.
START_DATE = "2014-12-31"
END_DATE = "2017-10-29"

# Критерии отбора магазинов из исходного ноутбука.
MIN_ACTIVE_DAYS = 100
MIN_TRANSACTIONS = 5000

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Не найден исходный CSV: {RAW_DATA_PATH.resolve()}\n"
        "Укажите корректный путь в RAW_DATA_PATH."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)


## 2. Загрузка и первичная проверка

In [3]:
df = pd.read_csv(
    RAW_DATA_PATH,
    sep=",",
    encoding="cp1251",
    dtype={"Zip Code": str},
)

print(f"Размер исходного набора: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
print(f"Память: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} МБ")
print("\nТипы данных:")
print(df.dtypes.to_string())
display(df.head())


Размер исходного набора: 12,591,077 строк × 24 столбцов
Память: 4,882.5 МБ

Типы данных:
Invoice/Item Number          str
Date                         str
Store Number               int64
Store Name                   str
Address                      str
City                         str
Zip Code                     str
Store Location               str
County Number            float64
County                       str
Category                 float64
Category Name                str
Vendor Number            float64
Vendor Name                  str
Item Number                int64
Item Description             str
Pack                       int64
Bottle Volume (ml)         int64
State Bottle Cost            str
State Bottle Retail          str
Bottles Sold               int64
Sale (Dollars)               str
Volume Sold (Liters)     float64
Volume Sold (Gallons)    float64


,Invoice/Item Number,Date,Store Number,Store Name,Address,City,Zip Code,Store Location,County Number,County,...,Item Number,Item Description,Pack,Bottle Volume (ml),State Bottle Cost,State Bottle Retail,Bottles Sold,Sale (Dollars),Volume Sold (Liters),Volume Sold (Gallons)
0,S29198800001,11/20/2015,2191,Keokuk Spirits,1013 MAIN,KEOKUK,52632,"1013 MAIN\nKEOKUK 52632\n(40.39978, -91.387531)",56.0,Lee,...,297,Templeton Rye w/Flask,6,750,$18.09,$27.14,6,$162.84,4.50,1.19
1,S29195400002,11/21/2015,2205,Ding's Honk And Holler,900 E WASHINGTON,CLARINDA,51632,"900 E WASHINGTON\nCLARINDA 51632\n(40.739238, ...",73.0,Page,...,297,Templeton Rye w/Flask,6,750,$18.09,$27.14,12,$325.68,9.00,2.38
2,S29050300001,11/16/2015,3549,Quicker Liquor Store,1414 48TH ST,FORT MADISON,52627,"1414 48TH ST\nFORT MADISON 52627\n(40.624226, ...",56.0,Lee,...,249,Disaronno Amaretto Cavalli Mignon 3-50ml Pack,20,150,$6.40,$9.60,2,$19.20,0.30,0.08
3,S28867700001,11/04/2015,2513,Hy-Vee Food Store #2 / Iowa City,812 S 1ST AVE,IOWA CITY,52240,812 S 1ST AVE\nIOWA CITY 52240\n,52.0,Johnson,...,237,Knob Creek w/ Crystal Decanter,3,1750,$35.55,$53.34,3,$160.02,5.25,1.39
4,S29050800001,11/17/2015,3942,Twin Town Liquor,104 HIGHWAY 30 WEST,TOLEDO,52342,"104 HIGHWAY 30 WEST\nTOLEDO 52342\n(41.985887,...",86.0,Tama,...,249,Disaronno Amaretto Cavalli Mignon 3-50ml Pack,20,150,$6.40,$9.60,2,$19.20,0.30,0.08


## 3. Приведение типов и отбор периода

In [4]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Некорректных дат:", int(df["Date"].isna().sum()))
print("Минимальная дата:", df["Date"].min())
print("Максимальная дата:", df["Date"].max())

text_cols = ["County", "Category Name", "Vendor Name", "City"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.upper()

money_cols = ["State Bottle Cost", "State Bottle Retail", "Sale (Dollars)"]
for col in money_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype("string").str.replace(r"[$,]", "", regex=True),
            errors="coerce",
        )

start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE)
df_period = df.loc[
    df["Date"].notna()
    & (df["Date"] > start_date)
    & (df["Date"] < end_date)
].copy()

print(f"Размер после отбора периода: {df_period.shape}")
print(f"Фактический период: {df_period['Date'].min()} — {df_period['Date'].max()}")


Некорректных дат: 0
Минимальная дата: 2012-01-03 00:00:00
Максимальная дата: 2017-10-31 00:00:00
Размер после отбора периода: (6327188, 24)
Фактический период: 2015-01-05 00:00:00 — 2017-10-27 00:00:00


## 4. Проверка распределения данных по городам

In [5]:
city_stats = (
    df_period.groupby("City", dropna=False)
    .agg(
        Store_Count=("Store Number", "nunique"),
        Transaction_Count=("Invoice/Item Number", "count"),
        Total_Bottles_Sold=("Bottles Sold", "sum"),
        Unique_Item_Count=("Item Number", "nunique"),
        Active_Days=("Date", "nunique"),
        Avg_Bottles_Per_Transaction=("Bottles Sold", "mean"),
    )
    .sort_values("Transaction_Count", ascending=False)
    .reset_index()
)

display(city_stats)

# Кандидаты по порогам из исходной ячейки № 3. Это диагностический вывод,
# а не автоматическая замена выбранного списка городов.
city_candidates = city_stats.loc[
    (city_stats["Store_Count"] >= 10)
    & (city_stats["Transaction_Count"] >= 100_000)
    & (city_stats["Unique_Item_Count"] >= 1500)
    & (city_stats["Active_Days"] >= 200)
].copy()

display(city_candidates)


,City,Store_Count,Transaction_Count,Total_Bottles_Sold,Unique_Item_Count,Active_Days,Avg_Bottles_Per_Transaction
0,DES MOINES,102,559323,5238892,4069,667,9.366488
1,CEDAR RAPIDS,89,428160,2973020,3207,535,6.943713
2,DAVENPORT,53,272104,2317725,3008,462,8.517791
3,WATERLOO,52,202551,1756413,2990,514,8.671461
4,COUNCIL BLUFFS,46,188237,1384742,2437,355,7.356375
...,...,...,...,...,...,...,...
399,MARQUETTE,1,74,336,69,2,4.540541
400,KANAWHA,1,70,81,48,6,1.157143
401,DOON,1,67,117,52,8,1.746269
402,WHITTEMORE,1,48,76,40,4,1.583333


,City,Store_Count,Transaction_Count,Total_Bottles_Sold,Unique_Item_Count,Active_Days,Avg_Bottles_Per_Transaction
0,DES MOINES,102,559323,5238892,4069,667,9.366488
1,CEDAR RAPIDS,89,428160,2973020,3207,535,6.943713
2,DAVENPORT,53,272104,2317725,3008,462,8.517791
3,WATERLOO,52,202551,1756413,2990,514,8.671461
4,COUNCIL BLUFFS,46,188237,1384742,2437,355,7.356375
6,IOWA CITY,32,183047,1471847,2831,392,8.040815
7,AMES,27,175945,1073461,3238,410,6.101117
8,WEST DES MOINES,34,171898,1244667,2795,581,7.240730
10,CEDAR FALLS,19,135172,773860,2758,384,5.725002
11,ANKENY,21,114441,777576,2194,580,6.794558


## 5. Фиксация выбранных городов

In [6]:
SELECTED_CITIES = [
    "DES MOINES",
    "CEDAR RAPIDS",
    "DAVENPORT",
    "WATERLOO",
    "COUNCIL BLUFFS",
    "WEST DES MOINES",
]

city_data = df_period.loc[df_period["City"].isin(SELECTED_CITIES)].copy()

print("Выбранные города, найденные в данных:")
print(sorted(city_data["City"].dropna().unique().tolist()))
print("\nГорода из списка, для которых нет строк:")
print(sorted(set(SELECTED_CITIES) - set(city_data["City"].dropna().unique())))
print("Размер данных выбранных городов:", city_data.shape)


Выбранные города, найденные в данных:
['CEDAR RAPIDS', 'COUNCIL BLUFFS', 'DAVENPORT', 'DES MOINES', 'WATERLOO', 'WEST DES MOINES']

Города из списка, для которых нет строк:
[]
Размер данных выбранных городов: (1822273, 24)


## 6. Проверка активности магазинов

In [7]:
store_stats = (
    city_data.groupby(["City", "Store Number"], dropna=False)
    .agg(
        Transaction_Count=("Invoice/Item Number", "count"),
        Unique_Item_Count=("Item Number", "nunique"),
        Active_Days=("Date", "nunique"),
        Total_Bottles_Sold=("Bottles Sold", "sum"),
    )
    .reset_index()
)

eligible_stores = store_stats.loc[
    (store_stats["Active_Days"] >= MIN_ACTIVE_DAYS)
    & (store_stats["Transaction_Count"] >= MIN_TRANSACTIONS)
].copy()

display(
    eligible_stores.groupby("City")
    .agg(
        Stores=("Store Number", "nunique"),
        Records=("Transaction_Count", "sum"),
        Median_Active_Days=("Active_Days", "median"),
    )
    .sort_values("Stores", ascending=False)
)

weekday_results = []
for city_name, group in city_data.groupby("City"):
    qualifying_ids = eligible_stores.loc[
        eligible_stores["City"] == city_name, "Store Number"
    ]
    for store_id in qualifying_ids:
        dates = (
            group.loc[group["Store Number"] == store_id, "Date"]
            .drop_duplicates()
            .sort_values()
        )
        intervals = dates.diff().dt.days.dropna()
        weekday_counts = dates.dt.day_name().value_counts()
        weekday_results.append({
            "City": city_name,
            "Store Number": store_id,
            "Active_Days": len(dates),
            "Median_Interval_Days": intervals.median(),
            "Most_Common_Weekday": weekday_counts.idxmax() if not weekday_counts.empty else None,
            "Most_Common_Weekday_Share": (
                weekday_counts.max() / len(dates) if len(dates) else np.nan
            ),
        })

weekday_check = pd.DataFrame(weekday_results)
print("Число магазинов в проверке:", len(weekday_check))
display(weekday_check)


,Stores,Records,Median_Active_Days
City,,,
DES MOINES,26,398600,159.5
CEDAR RAPIDS,22,324160,146.5
DAVENPORT,16,195558,146.0
COUNCIL BLUFFS,12,144789,147.5
WATERLOO,12,125552,146.5
WEST DES MOINES,6,116252,176.0


Число магазинов в проверке: 94


,City,Store Number,Active_Days,Median_Interval_Days,Most_Common_Weekday,Most_Common_Weekday_Share
0,CEDAR RAPIDS,2508,150,7.0,Tuesday,0.520000
1,CEDAR RAPIDS,2509,147,7.0,Tuesday,0.959184
2,CEDAR RAPIDS,2529,146,7.0,Tuesday,0.958904
3,CEDAR RAPIDS,2552,160,7.0,Tuesday,0.887500
4,CEDAR RAPIDS,2568,146,7.0,Wednesday,0.952055
...,...,...,...,...,...,...
89,WEST DES MOINES,2535,173,7.0,Monday,0.456647
90,WEST DES MOINES,2619,182,7.0,Tuesday,0.758242
91,WEST DES MOINES,2648,189,7.0,Monday,0.428571
92,WEST DES MOINES,3899,149,7.0,Monday,0.469799


## 7. Формирование `selected_city`

In [8]:
store_ids = eligible_stores["Store Number"].dropna().unique()
selected_city = city_data.loc[city_data["Store Number"].isin(store_ids)].copy()

print("Магазинов после фильтра:", len(store_ids))
print("Размер selected_city:", selected_city.shape)
print("\nМагазины по городам:")
display(
    selected_city.groupby("City")
    .agg(
        Stores=("Store Number", "nunique"),
        Records=("Invoice/Item Number", "count"),
        Active_Days=("Date", "nunique"),
    )
    .sort_values("Stores", ascending=False)
)


Магазинов после фильтра: 94
Размер selected_city: (1304911, 24)

Магазины по городам:


,Stores,Records,Active_Days
City,,,
DES MOINES,26,398600,667
CEDAR RAPIDS,22,324160,467
DAVENPORT,16,195558,448
COUNCIL BLUFFS,12,144789,335
WATERLOO,12,125552,447
WEST DES MOINES,6,116252,496


## 8. Справочники магазинов и товаров

In [9]:
store_cols = [
    "Store Number", "Store Name", "Address", "City", "Zip Code", "County",
]
store_cols = [col for col in store_cols if col in selected_city.columns]
store_lookup = (
    selected_city[store_cols]
    .drop_duplicates(subset="Store Number")
    .reset_index(drop=True)
)

item_cols = [
    "Item Number", "Item Description", "Category", "Category Name",
    "Vendor Number", "Vendor Name", "Pack", "Bottle Volume (ml)",
]
item_cols = [col for col in item_cols if col in selected_city.columns]
item_lookup = (
    selected_city[item_cols]
    .drop_duplicates(subset="Item Number")
    .reset_index(drop=True)
)

print("Справочник магазинов:", store_lookup.shape)
print("Уникальных магазинов:", store_lookup["Store Number"].nunique())
print("Справочник товаров:", item_lookup.shape)
print("Уникальных товаров:", item_lookup["Item Number"].nunique())


Справочник магазинов: (94, 6)
Уникальных магазинов: 94
Справочник товаров: (4249, 8)
Уникальных товаров: 4249


## 9. Формирование наборов для моделирования

In [10]:
model_data = selected_city[
    ["Date", "Store Number", "Item Number", "Volume Sold (Liters)"]
].copy()

# Week остаётся Period в рабочем ml_weekly и преобразуется в строку только при экспорте.
model_data["Week"] = model_data["Date"].dt.to_period("W")
ml_weekly = (
    model_data.groupby("Week", as_index=False)
    .agg(**{"Volume Sold (Liters)": ("Volume Sold (Liters)", "sum")})
    .sort_values("Week")
    .reset_index(drop=True)
)
ml_weekly["Year"] = ml_weekly["Week"].dt.start_time.dt.year

print("model_data:", model_data.shape)
print("ml_weekly:", ml_weekly.shape)
if not ml_weekly.empty:
    print("Диапазон недель:", ml_weekly["Week"].min(), "—", ml_weekly["Week"].max())
print("\nПропуски в недельном ряду:")
print(ml_weekly.isna().sum().to_string())
display(ml_weekly.head())
display(ml_weekly.tail())


model_data: (1304911, 5)
ml_weekly: (147, 3)
Диапазон недель: 2015-01-05/2015-01-11 — 2017-10-23/2017-10-29

Пропуски в недельном ряду:
Week                    0
Volume Sold (Liters)    0
Year                    0


,Week,Volume Sold (Liters),Year
0,2015-01-05/2015-01-11,85395.63,2015
1,2015-01-12/2015-01-18,107265.54,2015
2,2015-01-19/2015-01-25,48501.05,2015
3,2015-01-26/2015-02-01,85677.94,2015
4,2015-02-02/2015-02-08,89135.13,2015


,Week,Volume Sold (Liters),Year
142,2017-09-25/2017-10-01,14817.71,2017
143,2017-10-02/2017-10-08,20523.49,2017
144,2017-10-09/2017-10-15,18737.14,2017
145,2017-10-16/2017-10-22,18092.09,2017
146,2017-10-23/2017-10-29,19364.17,2017


## 10. Проверка непрерывности недельного ряда

In [11]:
if not ml_weekly.empty:
    first_week = ml_weekly["Week"].min().start_time
    last_week = ml_weekly["Week"].max().start_time
    expected_weeks = pd.date_range(first_week, last_week, freq="W-MON")
    actual_weeks = pd.DatetimeIndex(ml_weekly["Week"].dt.start_time)
    missing_weeks = expected_weeks.difference(actual_weeks)

    print("Ожидаемое количество недель:", len(expected_weeks))
    print("Фактическое количество недель:", len(actual_weeks))
    print("Пропущенных недель:", len(missing_weeks))
    if len(missing_weeks):
        print(missing_weeks)


Ожидаемое количество недель: 147
Фактическое количество недель: 147
Пропущенных недель: 0


## 11. Сохранение подготовленных данных

In [12]:
selected_city_path = DATA_DIR / "selected_city.parquet"
model_data_path = DATA_DIR / "model_data.parquet"
store_lookup_path = DATA_DIR / "store_lookup.csv"
item_lookup_path = DATA_DIR / "item_lookup.csv"
ml_weekly_path = DATA_DIR / "ml_weekly.csv"

selected_city.to_parquet(selected_city_path, index=False)
model_data.assign(Week=model_data["Week"].astype(str)).to_parquet(model_data_path, index=False)
store_lookup.to_csv(store_lookup_path, index=False, encoding="utf-8-sig")
item_lookup.to_csv(item_lookup_path, index=False, encoding="utf-8-sig")
ml_weekly.assign(Week=ml_weekly["Week"].astype(str)).to_csv(
    ml_weekly_path, index=False, encoding="utf-8-sig"
)

for output_path in [
    selected_city_path, model_data_path, store_lookup_path,
    item_lookup_path, ml_weekly_path,
]:
    print(f"Сохранено: {output_path} ({output_path.stat().st_size / 1024**2:.2f} МБ)")


Сохранено: data\processed\selected_city.parquet (27.43 МБ)
Сохранено: data\processed\model_data.parquet (4.32 МБ)
Сохранено: data\processed\store_lookup.csv (0.01 МБ)
Сохранено: data\processed\item_lookup.csv (0.38 МБ)
Сохранено: data\processed\ml_weekly.csv (0.01 МБ)
